Cloud-only data dependency: this notebook expects access to OncDRS/cloud data paths and will not run in a local-only environment.

# Binary NEPC: Generate Notes and Run the LLM

This notebook uses the same raw OncDRS note-reading and ICD-based cohort inference logic as the pipeline scripts.

Workflow:
1. Infer or specify the MRN cohort.
2. Read raw OncDRS notes.
3. Optionally write a compiled notes CSV and/or gzip bundle.
4. Build trigger-centered snippets.
5. Run the NEPC classifier on one patient.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "binary_NEPC" else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from binary_NEPC.run_NEPC_classifier import classify_patient
from shared.compile_prostate_notes import DEFAULT_ICD_SOURCE, derive_prostate_mrns
from shared.llm_helpers import (
    DEFAULT_OUTPUT_DIR,
    DEFAULT_RAW_TEXT_PATHS,
    NOTE_BUNDLE_FILENAME,
    PROSTATE_TEXT_CSV,
    build_client,
    build_patient_snippets,
    load_raw_text_notes,
    parse_mrn_values,
    resolve_raw_text_paths,
    write_note_bundle,
    write_notes_csv,
)

pd.set_option("display.max_colwidth", 200)

In [ ]:
# Parameters
# Leave MRNS empty to infer the full cohort from raw OncDRS ICDs.
MRNS = []
MRN_FILE = None

# Uses the raw OncDRS diagnosis source by default.
ICD_SOURCE = DEFAULT_ICD_SOURCE

# Defaults to the shared raw note locations from shared.llm_helpers.
RAW_TEXT_PATHS = list(DEFAULT_RAW_TEXT_PATHS)

# Output locations for optional compiled artifacts.
NOTES_CSV_PATH = PROSTATE_TEXT_CSV
NOTE_BUNDLE_PATH = DEFAULT_OUTPUT_DIR / NOTE_BUNDLE_FILENAME
WRITE_NOTES_CSV = False
WRITE_NOTE_BUNDLE = False

# Snippet / classification controls
MAX_NOTES_PER_PATIENT = 30
MODEL_NAME = "gpt-4o"
MAX_RETRIES = 3

# Optional: set this to a specific MRN to inspect and classify.
TARGET_MRN = None

In [ ]:
selected_mrns = set()

if MRNS:
    selected_mrns |= parse_mrn_values(MRNS)

if MRN_FILE is not None:
    mrn_file_path = Path(MRN_FILE)
    if not mrn_file_path.exists():
        raise FileNotFoundError(f"MRN file not found: {mrn_file_path}")
    file_mrns = pd.read_csv(mrn_file_path, header=None).iloc[:, 0].tolist()
    selected_mrns |= parse_mrn_values(file_mrns)

if not selected_mrns:
    selected_mrns = derive_prostate_mrns(ICD_SOURCE)
    print(f"Derived cohort from ICD source: {len(selected_mrns)} MRNs")
else:
    print(f"Using explicit MRN selection: {len(selected_mrns)} MRNs")

raw_text_paths = resolve_raw_text_paths(RAW_TEXT_PATHS)
print("ICD source:", ICD_SOURCE)
print("Raw note roots:")
for path in raw_text_paths:
    print("  ", path)

In [ ]:
notes_df = load_raw_text_notes(raw_text_paths, selected_mrns)
print(f"Loaded {len(notes_df):,} notes for {notes_df['DFCI_MRN'].nunique():,} patients")
display(notes_df.head())

In [ ]:
if WRITE_NOTES_CSV:
    standardized = write_notes_csv(NOTES_CSV_PATH, notes_df)
    print(f"Wrote notes CSV: {NOTES_CSV_PATH}")
    print(f"CSV rows: {len(standardized):,}")

if WRITE_NOTE_BUNDLE:
    write_note_bundle(
        NOTE_BUNDLE_PATH,
        notes_df,
        raw_text_paths=raw_text_paths,
        selected_mrns=selected_mrns,
    )
    print(f"Wrote note bundle: {NOTE_BUNDLE_PATH}")

In [ ]:
patient_snippets = build_patient_snippets(
    notes_df,
    max_notes_per_patient=MAX_NOTES_PER_PATIENT,
)

triggered_mrns = sorted(patient_snippets)
print(f"Patients with triggered snippets: {len(triggered_mrns):,}")

snippet_summary = pd.DataFrame(
    [
        {
            "DFCI_MRN": mrn,
            "num_snippets": len(snippets),
            "note_types": sorted({s['note_type'] for s in snippets}),
            "trigger_categories": sorted({c for s in snippets for c in s['trigger_categories']}),
        }
        for mrn, snippets in patient_snippets.items()
    ]
).sort_values(["num_snippets", "DFCI_MRN"], ascending=[False, True])

display(snippet_summary.head(20))

In [ ]:
if not triggered_mrns:
    raise ValueError("No trigger-bearing patients were found in the selected note set.")

mrn_to_review = int(TARGET_MRN) if TARGET_MRN is not None else int(triggered_mrns[0])
snippets = patient_snippets[mrn_to_review]

print(f"Reviewing MRN: {mrn_to_review}")
print(f"Snippets sent to model: {len(snippets)}")

snippet_df = pd.DataFrame(snippets)
display(snippet_df[["note_date", "note_type", "trigger_categories"]])
display(snippet_df[["note_date", "note_type", "snippet"]].head(5))

In [ ]:
payload_preview = {
    "patient_mrn": mrn_to_review,
    "notes": [
        {
            "note_date": s["note_date"],
            "note_type": s["note_type"],
            "trigger_categories": s["trigger_categories"],
            "note_text": s["snippet"],
        }
        for s in snippets
    ],
}

print(json.dumps(payload_preview, ensure_ascii=False, indent=2)[:6000])

In [ ]:
client = build_client()
result, error = classify_patient(
    client=client,
    model=MODEL_NAME,
    max_retries=MAX_RETRIES,
    mrn=mrn_to_review,
    snippets=snippets,
)

if error:
    raise RuntimeError(error)

result